# LLM-Assisted Personalized Fashion Recommendation Using User Preferences and Semantic Retrieval
**Authoritative Specification:** `End_to_End PipeLine.pdf`  
**Status:** Phase 5 — Final Integration, Evaluation & Project Deployment Complete

---
### Notebook Outline
1. **Setup** — Environment initialization, Colab detection, seed initialization & config loading
2. **Catalog** — Dataset ingestion, image validation, stratified sampling & 10,000 product frozen catalog
3. **Embeddings** — Text representation builder, all-MiniLM-L6-v2 embeddings & cosine semantic retrieval
4. **Profiles** — SQLite profile storage, product likes/feedback tracking & preference vector construction
5. **LLM Preference Extraction** — Qwen2.5-3B-Instruct preference extraction with Pydantic schema validation
6. **Ranking** — Multi-stage personalized ranking with strict constraint filtering, taste weighting & dismissal
7. **Explanations** — Evidence-based explanation generation with strict attribute grounding validation
8. **Interactive Interface** — Interactive multi-tab ipywidgets application with visual product cards & feedback
9. **Formal Evaluation & 4-System Comparative Study** — 40-query benchmark comparing Systems A, B, C, and D
10. **Persistence, Runtime Recovery & Project Packaging** — SQLite snapshot backup & sanitized export archive


## 1. Setup

In this section, we establish a reproducible runtime environment supporting both Google Colab (with Google Drive mounting) and local workstations. We detect Python and PyTorch environments, verify GPU availability, initialize workspace directories, install dependencies without conflicting with existing PyTorch installations, and configure global project paths with a fixed random seed of `42`.


In [ ]:
# 1.1 Colab Detection & Directory Initialization
import os
import sys
import shutil
from pathlib import Path

# Detect execution platform
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab environment.")
    from google.colab import drive
    drive_mount_path = Path("/content/drive")
    if not drive_mount_path.exists():
        drive.mount("/content/drive")
    # Project paths in Colab
    WORKSPACE_ROOT = Path("/content/drive/MyDrive/fashion_project")
    LOCAL_RUNTIME_ROOT = Path("/content/fashion_project")
else:
    print("[INFO] Running in Local Workstation environment.")
    WORKSPACE_ROOT = Path(".").resolve()
    LOCAL_RUNTIME_ROOT = WORKSPACE_ROOT

# Define canonical directory hierarchy
DATA_DIR = LOCAL_RUNTIME_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
IMAGES_DIR = RAW_DATA_DIR / "images"
ARTIFACTS_DIR = LOCAL_RUNTIME_ROOT / "artifacts"
BACKUPS_DIR = LOCAL_RUNTIME_ROOT / "backups"
LOGS_DIR = LOCAL_RUNTIME_ROOT / "logs"

for directory in [RAW_DATA_DIR, IMAGES_DIR, ARTIFACTS_DIR, BACKUPS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"[SUCCESS] Workspace initialized at: {LOCAL_RUNTIME_ROOT}")
print(f"  Artifacts directory: {ARTIFACTS_DIR}")
print(f"  Raw data directory:  {RAW_DATA_DIR}")


In [ ]:
# 1.2 System & Runtime Inspection
import platform

print("="*60)
print("RUNTIME HARDWARE & SOFTWARE SPECIFICATIONS")
print("="*60)
print(f"OS Platform:       {platform.platform()}")
print(f"Python Version:    {sys.version.split()[0]}")

# PyTorch & GPU Detection
try:
    import torch
    print(f"PyTorch Version:   {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available:    {cuda_available}")
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"Assigned GPU:      {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    else:
        print("Assigned GPU:      None (CPU Runtime)")
except ImportError:
    print("PyTorch Version:   Not Installed (will be loaded in PyTorch runtime)")

# System RAM Detection
def detect_ram():
    try:
        import psutil
        return psutil.virtual_memory().total / (1024**3)
    except Exception:
        pass
    if sys.platform == "win32":
        import ctypes
        class MEMORYSTATUSEX(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        stat = MEMORYSTATUSEX()
        stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
        return stat.ullTotalPhys / (1024**3)
    elif os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo") as f:
            for line in f:
                if "MemTotal" in line:
                    return int(line.split()[1]) / (1024**2)
    return 0.0

ram_gb = detect_ram()
print(f"System RAM:        {ram_gb:.2f} GB")

# Disk Space Detection
disk_total, disk_used, disk_free = shutil.disk_usage(".")
print(f"Disk Capacity:     {disk_total / (1024**3):.2f} GB total, {disk_free / (1024**3):.2f} GB free")
print("="*60)


In [ ]:
# 1.3 Dependency Installation (Colab / Local compatible)
# Preserves existing PyTorch installation as mandated by the implementation guide
REQUIRED_PACKAGES = [
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "pillow>=10.0.0",
    "pydantic>=2.10.0",
    "ipywidgets>=8.0.0"
]

if IN_COLAB:
    print("[INFO] Installing / verifying required packages in Colab...")
    get_ipython().system("pip install --quiet " + " ".join(REQUIRED_PACKAGES))
else:
    print("[INFO] Verifying packages in local environment...")
    import pandas, numpy, PIL, pydantic
    print(f"  pandas:  {pandas.__version__}")
    print(f"  numpy:   {numpy.__version__}")
    print(f"  pillow:  {PIL.__version__}")
    print(f"  pydantic:{pydantic.__version__}")


In [ ]:
# 1.4 Configuration Management & Random Seed Initialization
import json
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CONFIG_PATH = LOCAL_RUNTIME_ROOT / "config.json"

config = {
    "project_title": "LLM-Assisted Personalized Fashion Recommendation Using User Preferences and Semantic Retrieval",
    "current_phase": 1,
    "random_seed": RANDOM_SEED,
    "dataset": {
        "source": "https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small",
        "styles_csv": str(RAW_DATA_DIR / "styles.csv"),
        "images_dir": str(IMAGES_DIR)
    },
    "paths": {
        "catalog_csv": str(ARTIFACTS_DIR / "catalog.csv"),
        "catalog_ids_json": str(ARTIFACTS_DIR / "catalog_ids.json"),
        "data_audit_json": str(ARTIFACTS_DIR / "data_audit.json"),
        "catalog_manifest_json": str(ARTIFACTS_DIR / "catalog_manifest.json")
    },
    "planned_models": {
        "language_model": "Qwen/Qwen2.5-3B-Instruct",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"[SUCCESS] Global configuration saved to: {CONFIG_PATH}")


## 2. Catalog

In this section, we acquire the Kaggle dataset (`fashion-product-images-small`), inspect and validate all 10 raw attributes, perform Pillow-based image file decoding checks, apply strict filtering (keeping only Apparel and Footwear, eliminating records with missing attributes or unusable images), and draw a frozen 10,000-item stratified sample across `articleType` groups using seed `42`.


In [ ]:
# 2.1 Dataset Acquisition & Verification
import zipfile

styles_csv_path = RAW_DATA_DIR / "styles.csv"
images_dir_path = IMAGES_DIR

# Kaggle download logic with Colab Secrets support and manual upload fallback
if not styles_csv_path.exists() or len(os.listdir(images_dir_path)) == 0:
    print("[INFO] Dataset not found in raw data directory. Initiating acquisition...")
    
    archive_path = RAW_DATA_DIR / "fashion-product-images-small.zip"
    if not archive_path.exists():
        archive_candidates = [
            Path("fashion_project/data/raw/fashion-product-images-small.zip"),
            Path("data/raw/fashion-product-images-small.zip")
        ]
        for c in archive_candidates:
            if c.exists():
                archive_path = c
                break
                
    if archive_path.exists():
        print(f"[INFO] Extracting archive from {archive_path}...")
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(RAW_DATA_DIR)
        print("[SUCCESS] Extraction completed.")
    elif IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            print("[INFO] Authenticating via Kaggle API from Colab Secrets...")
            get_ipython().system("kaggle datasets download -d paramaggarwal/fashion-product-images-small -p /content/fashion_project/data/raw --unzip")
        except Exception as e:
            print("[WARNING] Kaggle credentials not found in Colab Secrets.")
            print("Please configure KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets or upload archive manually.")
    else:
        print("[NOTICE] Please ensure styles.csv and images/ are present in data/raw/.")

# Final check
assert styles_csv_path.exists(), f"styles.csv not found at {styles_csv_path}"
assert images_dir_path.exists(), f"images directory not found at {images_dir_path}"
print(f"[SUCCESS] styles.csv verified: {os.path.getsize(styles_csv_path):,} bytes")
print(f"[SUCCESS] images directory verified: {len(os.listdir(images_dir_path)):,} files found")


In [ ]:
# 2.2 Dataset Ingestion & Column Validation
import csv
import pandas as pd

EXPECTED_COLUMNS = [
    "id", "gender", "masterCategory", "subCategory", "articleType",
    "baseColour", "season", "year", "usage", "productDisplayName"
]
NA_STRINGS = {"", "NA", "NaN", "null", "none"}

malformed_records = []
valid_rows = []

with open(styles_csv_path, mode="r", encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert header == EXPECTED_COLUMNS, f"Header mismatch! Expected {EXPECTED_COLUMNS}, got {header}"
    
    for line_idx, row in enumerate(reader, start=2):
        if len(row) != 10:
            malformed_records.append({
                "line_number": line_idx,
                "field_count": len(row),
                "id": row[0] if len(row) > 0 else f"unknown_{line_idx}",
                "reason": "malformed_csv_token_count_mismatch"
            })
        else:
            cleaned = [val.strip() if val.strip() not in NA_STRINGS else None for val in row]
            valid_rows.append(cleaned)

df_raw = pd.DataFrame(valid_rows, columns=EXPECTED_COLUMNS)
df_raw["id"] = df_raw["id"].astype(str).str.strip()

print(f"Total rows parsed:       {len(valid_rows) + len(malformed_records):,}")
print(f"Valid 10-column records: {len(df_raw):,}")
print(f"Malformed records:       {len(malformed_records):,} (recorded in data audit)")
print(f"Duplicate product IDs:   {df_raw['id'].duplicated().sum()}")


In [ ]:
# 2.3 Exploratory Data Analysis & Attribute Inspection
print("="*60)
print("RAW DATASET ATTRIBUTE INSPECTION")
print("="*60)

# Missing Values Summary
missing_series = df_raw.isna().sum()
print("Missing values per column:")
for col, cnt in missing_series.items():
    print(f"  {col:<20}: {cnt:>5} ({cnt/len(df_raw)*100:>5.2f}%)")

print("
Distribution by masterCategory:")
print(df_raw["masterCategory"].value_counts(dropna=False))

print("
Top 10 articleTypes:")
print(df_raw["articleType"].value_counts(dropna=False).head(10))

print("
Top 10 baseColours:")
print(df_raw["baseColour"].value_counts(dropna=False).head(10))

print("
Distribution by Gender:")
print(df_raw["gender"].value_counts(dropna=False))

print("
Distribution by Usage:")
print(df_raw["usage"].value_counts(dropna=False))
print("="*60)


In [ ]:
# 2.4 Multithreaded Image Validation with Pillow
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

existing_image_files = set(os.listdir(images_dir_path))
all_raw_ids = df_raw["id"].tolist()

missing_image_ids = [pid for pid in all_raw_ids if f"{pid}.jpg" not in existing_image_files]
existing_image_ids = [pid for pid in all_raw_ids if f"{pid}.jpg" in existing_image_files]

def check_image_file(pid):
    p = images_dir_path / f"{pid}.jpg"
    try:
        with Image.open(p) as img:
            img.verify()
        with Image.open(p) as img:
            img.load()
        return pid, True, None
    except Exception as e:
        return pid, False, str(e)

print(f"[INFO] Validating {len(existing_image_ids):,} images with Pillow (32 threads)...")
with ThreadPoolExecutor(max_workers=32) as executor:
    image_check_results = list(executor.map(check_image_file, existing_image_ids))

valid_image_ids = set(pid for pid, ok, err in image_check_results if ok)
corrupted_images = [{"id": pid, "error": err} for pid, ok, err in image_check_results if not ok]

print(f"  Valid decodable images:   {len(valid_image_ids):,}")
print(f"  Missing image files:      {len(missing_image_ids):,} -> {missing_image_ids}")
print(f"  Corrupted image files:    {len(corrupted_images):,}")


In [ ]:
# 2.5 Filtering, Stratified Sampling & Frozen Catalog Generation
# Rule 1: Keep masterCategory in Apparel & Footwear
af_mask = df_raw["masterCategory"].isin(["Apparel", "Footwear"])
af_df = df_raw[af_mask].copy()

# Rule 2: Exclude missing images
has_image_mask = af_df["id"].isin(valid_image_ids)
af_with_images = af_df[has_image_mask].copy()

# Rule 3: Exclude records with missing attributes in any of the 10 columns
complete_metadata_mask = ~af_with_images.isna().any(axis=1)
clean_candidates = af_with_images[complete_metadata_mask].copy()

# Rule 4: Eliminate any duplicate IDs
clean_candidates = clean_candidates.drop_duplicates(subset=["id"]).sort_values(by="id").reset_index(drop=True)

total_clean_candidates = len(clean_candidates)
print(f"Total Apparel & Footwear:     {len(af_df):,}")
print(f"Eligible clean candidates:    {total_clean_candidates:,}")

# Stratified sampling across articleType capped at 10,000 items
TARGET_SIZE = min(10000, total_clean_candidates)
groups = clean_candidates.groupby("articleType")

allocations = {}
for name, group in groups:
    share = int(np.floor(len(group) * TARGET_SIZE / total_clean_candidates))
    allocations[name] = max(1, share)

diff = TARGET_SIZE - sum(allocations.values())
sorted_groups = sorted(allocations.keys(), key=lambda k: len(groups.get_group(k)), reverse=True)
for i in range(abs(diff)):
    k = sorted_groups[i % len(sorted_groups)]
    if diff > 0:
        allocations[k] += 1
    else:
        allocations[k] -= 1

sampled_subsets = []
for name, group in groups:
    s = group.sample(n=allocations[name], random_state=RANDOM_SEED)
    sampled_subsets.append(s)

catalog_df = pd.concat(sampled_subsets, ignore_index=True)
catalog_df = catalog_df.sort_values(by="id").reset_index(drop=True)
catalog_ids = catalog_df["id"].tolist()

# Export artifacts
catalog_csv_path = ARTIFACTS_DIR / "catalog.csv"
catalog_ids_path = ARTIFACTS_DIR / "catalog_ids.json"

catalog_df.to_csv(catalog_csv_path, index=False)
with open(catalog_ids_path, "w", encoding="utf-8") as f:
    json.dump(catalog_ids, f, indent=2)

print(f"[SUCCESS] Final frozen catalog exported to: {catalog_csv_path}")
print(f"  Total items:              {len(catalog_df):,}")
print(f"  Distinct article types:   {catalog_df['articleType'].nunique():,}")
print(f"  Product IDs exported:     {catalog_ids_path}")


In [ ]:
# 2.6 Visual Verification: Display Sample Catalog Products
import matplotlib.pyplot as plt

sample_subset = catalog_df.sample(n=5, random_state=42)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (_, row) in zip(axes, sample_subset.iterrows()):
    img_path = IMAGES_DIR / f"{row['id']}.jpg"
    try:
        im = Image.open(img_path)
        ax.imshow(im)
    except Exception:
        ax.text(0.5, 0.5, "Image Error", ha="center")
    ax.axis("off")
    title = f"{row['articleType']} ({row['baseColour']})\n{row['usage']}\nID: {row['id']}"
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()


## 3. Embeddings

In this section, we implement **Phase 2: Product Text Embeddings & Semantic Retrieval** following `End_to_End PipeLine.pdf` (Steps 3 & 4):
1. **Catalog Verification:** Load the frozen 10,000-item catalog and verify string ID alignment.
2. **Product Text Representation:** Generate structured descriptions using recorded attributes without fabricating unrecorded features.
3. **Embedding Model Initialization:** Load `sentence-transformers/all-MiniLM-L6-v2` once with device detection.
4. **Batch Embedding Generation & Persistence:** Encode descriptions into normalized 384-dimensional float32 vectors in batches of 64, save to `artifacts/embeddings.npy`, and record SHA-256 hashes in `artifacts/embedding_manifest.json`.
5. **Embedding Integrity & Reloading:** Verify assertions (`len(catalog) == len(catalog_ids) == len(embeddings)`, dimension 384, finite values, unit L2 norms).
6. **Semantic Retrieval Engine:** Retrieve top-5 candidates using pure NumPy vector dot-product similarity (cosine similarity).
7. **Visual Retrieval Demonstration:** Display retrieved product cards with images, metadata, and retrieval scores.


In [ ]:
# 3.1 Load Frozen Catalog & Build Product Text Representation
import json
import pandas as pd
from pathlib import Path

# Paths
catalog_csv_path = ARTIFACTS_DIR / "catalog.csv"
catalog_ids_path = ARTIFACTS_DIR / "catalog_ids.json"
desc_csv_path = ARTIFACTS_DIR / "product_descriptions.csv"

# 1. Load catalog and ordered ID manifest
catalog = pd.read_csv(catalog_csv_path, dtype={"id": str})
with open(catalog_ids_path, "r", encoding="utf-8") as f:
    catalog_ids = json.load(f)

assert len(catalog) == 10000, f"Expected 10,000 items, got {len(catalog)}"
assert catalog["id"].tolist() == catalog_ids, "Catalog ID sequence mismatch!"
print(f"[SUCCESS] Loaded frozen catalog with {len(catalog):,} products.")

# 2. Build or load product descriptions
def build_product_description(row):
    def _val(col, fallback="Unknown"):
        v = row.get(col)
        return str(v).strip() if pd.notna(v) and str(v).strip() not in ("", "NA", "null") else fallback
    return (
        f"Name: {_val('productDisplayName')}\n"
        f"Type: {_val('articleType')} | Color: {_val('baseColour')} | Usage: {_val('usage')}\n"
        f"Category: {_val('masterCategory')} | Season: {_val('season')}"
    )

if desc_csv_path.exists():
    print(f"[INFO] Loading existing product descriptions from {desc_csv_path}...")
    desc_df = pd.read_csv(desc_csv_path, dtype={"id": str})
    descriptions = desc_df["product_description"].tolist()
else:
    print("[INFO] Generating product descriptions from catalog metadata...")
    descriptions = [build_product_description(row) for _, row in catalog.iterrows()]
    desc_df = pd.DataFrame({
        "id": catalog["id"].astype(str).tolist(),
        "productDisplayName": catalog["productDisplayName"].tolist(),
        "product_description": descriptions
    })
    desc_df.to_csv(desc_csv_path, index=False)
    print(f"[SUCCESS] Saved {len(desc_df):,} descriptions to {desc_csv_path}")

print(f"\nSample Description (ID: {catalog.iloc[0]['id']}):")
print("-" * 50)
print(descriptions[0])
print("-" * 50)


In [ ]:
# 3.2 Initialize Embedding Model (sentence-transformers/all-MiniLM-L6-v2)
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Initializing '{MODEL_NAME}' on device: {device}...")

embedding_model = SentenceTransformer(MODEL_NAME, device=device)
embedding_model.eval()  # Inference mode

emb_dim = embedding_model.get_sentence_embedding_dimension()
print(f"[SUCCESS] Model initialized successfully.")
print(f"  Embedding dimension: {emb_dim} (Expected: 384)")
print(f"  Device:              {device}")


In [ ]:
# 3.3 Batch Embedding Generation, Normalization & Persistence
import numpy as np
import hashlib
import datetime

embeddings_npy_path = ARTIFACTS_DIR / "embeddings.npy"
embedding_manifest_path = ARTIFACTS_DIR / "embedding_manifest.json"

BATCH_SIZE = 64

def compute_file_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536 * 16):
            h.update(chunk)
    return h.hexdigest()

if embeddings_npy_path.exists() and embedding_manifest_path.exists():
    print(f"[INFO] Loading saved embedding matrix from {embeddings_npy_path}...")
    embeddings = np.load(embeddings_npy_path)
    with open(embedding_manifest_path, "r", encoding="utf-8") as f:
        emb_manifest = json.load(f)
    print(f"[SUCCESS] Reloaded embedding matrix. Shape: {embeddings.shape}")
else:
    print(f"[INFO] Computing embeddings for {len(descriptions):,} products (batch_size={BATCH_SIZE})...")
    embeddings = embedding_model.encode(
        descriptions,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    
    np.save(embeddings_npy_path, embeddings)
    print(f"[SUCCESS] Saved embeddings matrix to: {embeddings_npy_path}")
    
    # Save manifest
    emb_sha256 = compute_file_sha256(embeddings_npy_path)
    emb_manifest = {
        "project_title": "LLM-Assisted Personalized Fashion Recommendation Using User Preferences and Semantic Retrieval",
        "phase": 2,
        "creation_timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "model": {"name": MODEL_NAME, "dimension": int(embeddings.shape[1]), "normalized": True},
        "storage": {"file": "embeddings.npy", "dtype": str(embeddings.dtype), "shape": list(embeddings.shape), "sha256": emb_sha256},
        "catalog_size": len(catalog),
        "batch_size": BATCH_SIZE
    }
    with open(embedding_manifest_path, "w", encoding="utf-8") as f:
        json.dump(emb_manifest, f, indent=2)
    print(f"[SUCCESS] Saved embedding manifest to: {embedding_manifest_path}")

# Rigorous assertions mandated by End_to_End PipeLine.pdf
assert len(catalog) == len(catalog_ids) == len(embeddings), "Catalog / ID / Embedding length mismatch!"
assert embeddings.shape[1] == 384, f"Expected 384 dimensions, got {embeddings.shape[1]}"
assert catalog["id"].tolist() == catalog_ids, "Catalog ID list does not match CSV order!"
assert embeddings.dtype == np.float32, f"Expected float32, got {embeddings.dtype}"
assert np.all(np.isfinite(embeddings)), "Matrix contains NaN or Inf!"

norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-3), f"Embeddings not normalized: min={norms.min()}, max={norms.max()}"
print(f"[VERIFIED] All embedding integrity assertions passed cleanly (shape: {embeddings.shape}, dtype: {embeddings.dtype}).")


In [ ]:
# 3.4 Semantic Retrieval Engine (Vector Dot Product Cosine Similarity)
def retrieve_products(query, model, catalog, embeddings, top_k=5, images_dir=IMAGES_DIR):
    """
    Retrieve top_k products matching query using NumPy dot product cosine similarity.
    """
    if not query or not query.strip():
        return []
        
    actual_k = min(max(1, top_k), len(catalog))
    
    # Encode and normalize query
    q_vec = model.encode(query.strip(), normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    if q_vec.ndim == 1:
        q_vec = q_vec.reshape(1, -1)
        
    # Cosine similarity via dot product (both vectors are unit-normalized)
    scores = np.dot(embeddings, q_vec.T).squeeze()
    top_indices = np.argsort(scores)[::-1][:actual_k]
    
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        row = catalog.iloc[idx]
        pid = str(row["id"])
        img_p = Path(images_dir) / f"{pid}.jpg"
        results.append({
            "rank": rank,
            "id": pid,
            "productDisplayName": str(row["productDisplayName"]),
            "articleType": str(row["articleType"]),
            "baseColour": str(row["baseColour"]),
            "usage": str(row["usage"]),
            "masterCategory": str(row["masterCategory"]),
            "season": str(row["season"]),
            "retrieval_score": float(scores[idx]),
            "image_path": str(img_p),
            "image_exists": img_p.exists()
        })
    return results

print("[SUCCESS] Semantic retrieval function defined.")


In [ ]:
# 3.5 Interactive Visual Demonstration of Semantic Retrieval
import matplotlib.pyplot as plt
from PIL import Image

sample_query = "casual summer cotton t-shirt for men"
print(f"Search Query: '{sample_query}'\n" + "="*70)

top_results = retrieve_products(sample_query, embedding_model, catalog, embeddings, top_k=5)

for r in top_results:
    print(f"[{r['rank']}] Score: {r['retrieval_score']:.4f} | ID: {r['id']:<6} | {r['productDisplayName']}")
    print(f"     Type: {r['articleType']} | Color: {r['baseColour']} | Usage: {r['usage']} | Season: {r['season']}")

# Plot top 5 retrieved items
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle(f"Top 5 Semantic Retrieval Results for: '{sample_query}'", fontsize=14, y=1.05)

for ax, r in zip(axes, top_results):
    try:
        im = Image.open(r["image_path"])
        ax.imshow(im)
    except Exception:
        ax.text(0.5, 0.5, "Image Error", ha="center")
    ax.axis("off")
    title = f"#{r['rank']} (Score: {r['retrieval_score']:.3f})\n{r['articleType']} - {r['baseColour']}\nID: {r['id']}"
    ax.set_title(title, fontsize=9)

plt.tight_layout()
plt.show()


## 4. Profiles

In this section, we implement **Phase 3: User Profiles, Explicit Constraints & Product Feedback** following `End_to_End PipeLine.pdf` (Steps 5 & 7):

1. **SQLite Profile Storage:** Persistent local relational store at `backups/profiles.sqlite` tracking anonymous profiles, explicit preferences, active request constraints, and interaction feedback.
2. **Preference vs. Active Constraints Separation:** Long-term saved preferences (e.g. favorite colors and categories) remain strictly decoupled from short-term active request constraints (e.g. current query filters).
3. **Product Feedback (Likes & Dismissals):** Idempotent recording of user product likes and dismissals. Dismissed items are strictly filtered out of future recommendations.
4. **User Preference Vector Construction:** Normalized 384-dimensional vector computed as the mean embedding of liked catalog products (with explicit preference fallback and query-only fallback).
5. **Cross-Profile Isolation:** Independent user state management ensuring one user's preferences and feedback never contaminate another user's session.


In [ ]:
# 4.1 SQLite Database Setup & Anonymous Profile Creation
import json
import sqlite3
import pandas as pd
from pathlib import Path
from profile_manager import (
    init_db,
    create_profile,
    get_profile,
    update_profile,
    set_active_constraints,
    get_active_constraints,
    DEFAULT_DB_PATH
)

# Initialize SQLite database
DB_PATH = BACKUPS_DIR / "profiles.sqlite"
init_db(str(DB_PATH))
print(f"[INFO] SQLite database initialized at: {DB_PATH}")

# 1. Create Profile A (Casual Apparel Enthusiast)
profile_id_a = create_profile(
    preferences={
        "preferred_categories": ["Apparel"],
        "preferred_colors": ["Navy Blue", "Black"]
    },
    db_path=str(DB_PATH)
)

# Set initial active request constraints for Profile A
set_active_constraints(
    profile_id_a,
    constraints={
        "articleType": ["Shirts"],
        "baseColour": ["Navy Blue"],
        "usage": ["Casual"]
    },
    db_path=str(DB_PATH)
)

profile_a = get_profile(profile_id_a, db_path=str(DB_PATH))
active_cons_a = get_active_constraints(profile_id_a, db_path=str(DB_PATH))

print(f"[SUCCESS] Profile A created: {profile_id_a}")
print(f"  Saved Preferences:  {profile_a['preferences']}")
print(f"  Active Constraints: {active_cons_a}")

# 2. Create Profile B (Sports Footwear Enthusiast) for profile isolation testing
profile_id_b = create_profile(
    preferences={
        "preferred_categories": ["Footwear"],
        "preferred_colors": ["White", "Black"]
    },
    db_path=str(DB_PATH)
)
print(f"[SUCCESS] Profile B created: {profile_id_b}")


In [ ]:
# 4.2 Product Feedback (Likes & Idempotent Feedback Tracking)
from profile_manager import save_feedback, get_feedback
import matplotlib.pyplot as plt
from PIL import Image

# Identify 3 real catalog items matching Profile A's taste (Navy Blue / Black Casual Shirts)
matching_sample = catalog[
    (catalog["articleType"] == "Shirts") & 
    (catalog["baseColour"].isin(["Navy Blue", "Black"])) &
    (catalog["usage"] == "Casual")
].head(3)

sample_liked_ids = matching_sample["id"].tolist()
print(f"[INFO] Recording likes for Profile A on catalog products: {sample_liked_ids}")

# Save feedback with idempotency
for pid in sample_liked_ids:
    save_feedback(profile_id_a, pid, "like", db_path=str(DB_PATH))

# Test idempotency (repeated like does not duplicate rows)
save_feedback(profile_id_a, sample_liked_ids[0], "like", db_path=str(DB_PATH))

fb_a = get_feedback(profile_id_a, db_path=str(DB_PATH))
print(f"[SUCCESS] Profile A Likes count: {len(fb_a['likes'])} (Verified idempotent)")

# Display liked products
fig, axes = plt.subplots(1, len(sample_liked_ids), figsize=(12, 3.5))
fig.suptitle(f"Profile A Liked Products ({profile_id_a})", fontsize=12, y=1.05)

for ax, (_, row) in zip(axes, matching_sample.iterrows()):
    im_p = IMAGES_DIR / f"{row['id']}.jpg"
    try:
        im = Image.open(im_p)
        ax.imshow(im)
    except Exception:
        ax.text(0.5, 0.5, "Image Error", ha="center")
    ax.axis("off")
    ax.set_title(f"{row['productDisplayName'][:20]}...\nID: {row['id']} | {row['baseColour']}", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# 4.3 User Preference Vector Construction
from profile_manager import build_user_vector
import numpy as np

# 1. Generate user preference vector for Profile A from liked items
user_vec_a = build_user_vector(
    profile_id=profile_id_a,
    embeddings=embeddings,
    catalog_ids=catalog_ids,
    catalog=catalog,
    model=embedding_model,
    db_path=str(DB_PATH)
)

assert user_vec_a is not None, "Failed to build user vector for Profile A"
assert user_vec_a.shape == (384,), f"Expected shape (384,), got {user_vec_a.shape}"
assert np.isclose(np.linalg.norm(user_vec_a), 1.0, atol=1e-3), "Vector not unit normalized!"
print(f"[SUCCESS] User vector generated for Profile A:")
print(f"  Shape:     {user_vec_a.shape}")
print(f"  Dtype:     {user_vec_a.dtype}")
print(f"  L2 Norm:   {np.linalg.norm(user_vec_a):.5f}")

# 2. Test fallback for Profile B (No likes yet -> fallback to explicit preferences)
user_vec_b = build_user_vector(
    profile_id=profile_id_b,
    embeddings=embeddings,
    catalog_ids=catalog_ids,
    catalog=catalog,
    model=embedding_model,
    db_path=str(DB_PATH)
)
print(f"[SUCCESS] User vector generated for Profile B via explicit preference fallback:")
print(f"  Shape:     {user_vec_b.shape}")
print(f"  L2 Norm:   {np.linalg.norm(user_vec_b):.5f}")


## 5. LLM Preference Extraction

In this section, we implement **Phase 4: Qwen-Powered Conversational Preference Extraction & Pydantic Validation** following `End_to_End PipeLine.pdf` (Step 6, Page 4):

1. **Qwen2.5-3B-Instruct Model & Tokenizer:** Initialized in eval mode with deterministic decoding and official chat template (`<|im_start|>` / `<|im_end|>`).
2. **Strict JSON Schema & Operations:** Proposes structured state updates supporting `set` (replacement), `add` (union), `remove` (subtraction), `clear` (empty list), and `reset` (clearing active session constraints).
3. **Pydantic Validation & Catalog Domain Alignment:** Validates proposed fields, operations, and attribute values against the frozen catalog with deterministic synonym mapping (`navy` -> `Navy Blue`, etc.). Invented categories/colors are strictly rejected.
4. **Multi-Turn Context Preservation:** Distinguishes immediate corrections from previous compatible constraints, preserving existing state across conversational turns.
5. **Unsupported Request Handling:** Safely isolates unrecorded attributes (size, price, brand, fit, material) into an `unsupported` list.


In [ ]:
# 5.1 LLM Tokenizer & Hardware Inspection
from engines.preference_extractor import get_hardware_info, load_llm

hw_info = get_hardware_info()
print(f"[INFO] Hardware Environment:")
print(f"  Device:        {hw_info['device_name']}")
print(f"  CUDA Avail:    {hw_info['cuda_available']}")
print(f"  VRAM (GB):     {hw_info['vram_gb']}")
print(f"  RAM Avail (GB):{hw_info['ram_avail_gb']}")
print(f"  PyTorch:       {hw_info['torch_version']}")

# Attempt Qwen loading if environment supports it (CUDA or high RAM)
try:
    llm_tokenizer, llm_model = load_llm()
    print(f"[SUCCESS] Qwen2.5-3B-Instruct model and tokenizer loaded.")
except Exception as e:
    print(f"[NOTICE] Qwen model weights skipped on constrained local CPU ({e}).")
    print(f"         Using tokenizer and deterministic validated preference extractor.")
    from transformers import AutoTokenizer
    llm_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
    llm_model = None


In [ ]:
# 5.2 Multi-Turn Conversational Preference Extraction & Pydantic Validation
from engines.preference_extractor import (
    extract_update,
    validate_and_apply,
    get_allowed_catalog_values
)

allowed_vals = get_allowed_catalog_values()
conversation_state = {}

# Conversation Turn 1: Initial request
turn1_msg = "Show me casual shirts"
prop1 = extract_update(turn1_msg, conversation_state, allowed_vals, model=llm_model, tokenizer=llm_tokenizer)
ok1, conversation_state, err1 = validate_and_apply(conversation_state, prop1, allowed_vals)
print(f"Turn 1: '{turn1_msg}'")
print(f"  Proposed: {prop1.model_dump()}")
print(f"  State:    {conversation_state}\n")

# Conversation Turn 2: Color refinement
turn2_msg = "Black instead"
prop2 = extract_update(turn2_msg, conversation_state, allowed_vals, model=llm_model, tokenizer=llm_tokenizer)
ok2, conversation_state, err2 = validate_and_apply(conversation_state, prop2, allowed_vals)
print(f"Turn 2: '{turn2_msg}'")
print(f"  Proposed: {prop2.model_dump()}")
print(f"  State:    {conversation_state} (Preserved Shirts & Casual)\n")

# Conversation Turn 3: Negation / Excluded color
turn3_msg = "Avoid red"
prop3 = extract_update(turn3_msg, conversation_state, allowed_vals, model=llm_model, tokenizer=llm_tokenizer)
ok3, conversation_state, err3 = validate_and_apply(conversation_state, prop3, allowed_vals)
print(f"Turn 3: '{turn3_msg}'")
print(f"  Proposed: {prop3.model_dump()}")
print(f"  State:    {conversation_state}\n")

# Conversation Turn 4: Unsupported attribute request
turn4_msg = "Show me shirts in size Large under $50 by Nike"
prop4 = extract_update(turn4_msg, conversation_state, allowed_vals, model=llm_model, tokenizer=llm_tokenizer)
print(f"Turn 4: '{turn4_msg}'")
print(f"  Unsupported Attributes Isolated: {prop4.unsupported}")
print(f"  Valid State Preserved:           {conversation_state}")


## 6. Ranking

In this section, we implement **Phase 3: Candidate Filtering & Personalized Recommendation Ranking** following `End_to_End PipeLine.pdf` (Steps 5 & 7):

1. **Strict Hard Constraints Filtering:** Hard constraints (e.g. `articleType`, `baseColour`, `usage`, `masterCategory`, `exclude_colors`) are applied *before* ranking.
2. **Dismissed Product Removal:** Products dismissed by the user are strictly filtered out.
3. **Personalized Hybrid Ranking Formula:**
   $$\text{score}(\text{item}) = 0.8 \times \cos(q, \text{item\_vec}) + 0.2 \times \cos(u, \text{item\_vec})$$
   When the user vector $u$ is unavailable, the ranker falls back to query-only similarity $\text{score}(\text{item}) = \cos(q, \text{item\_vec})$.
4. **Deterministic Tie-Breaking:** Scores are stably ranked descending, breaking ties ascending by product ID.


In [ ]:
# 6.1 Personalized Recommendation Execution
from personalized_ranker import filter_candidates, recommend_personalized
import matplotlib.pyplot as plt
from PIL import Image

query = "casual comfortable cotton shirt"
constraints = {
    "articleType": ["Shirts"],
    "usage": ["Casual"],
    "exclude_colors": ["Red"]
}

print(f"Executing Personalized Recommendation for Profile A ({profile_id_a})...")
print(f"  Query:       '{query}'")
print(f"  Constraints: {constraints}\n")

rec_response = recommend_personalized(
    profile_id=profile_id_a,
    query=query,
    constraints=constraints,
    top_k=5,
    model=embedding_model,
    catalog=catalog,
    embeddings=embeddings,
    catalog_ids=catalog_ids,
    db_path=str(DB_PATH),
    images_dir=str(IMAGES_DIR)
)

print(f"Status: {rec_response['status']} | Total eligible items: {rec_response['total_eligible']:,}")
print(f"Has personalized vector: {rec_response['has_personalized_vector']}\n")

results = rec_response["results"]
for r in results:
    print(f"[{r['rank']}] Score: {r['score']:.4f} (Query: {r['query_similarity']:.4f}, Profile: {r['profile_similarity']:.4f}) | ID: {r['id']:<6}")
    print(f"     {r['productDisplayName']} ({r['baseColour']}, {r['articleType']})")

# Visual Display of Recommendations
fig, axes = plt.subplots(1, len(results), figsize=(18, 4))
fig.suptitle(f"Top {len(results)} Personalized Recommendations for Profile A", fontsize=14, y=1.05)

for ax, r in zip(axes, results):
    try:
        im = Image.open(r["image_path"])
        ax.imshow(im)
    except Exception:
        ax.text(0.5, 0.5, "Image Error", ha="center")
    ax.axis("off")
    title = f"#{r['rank']} (Score: {r['score']:.3f})\n{r['productDisplayName'][:18]}...\nColor: {r['baseColour']} | ID: {r['id']}"
    ax.set_title(title, fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# 6.2 Product Dismissal & Immediate Candidate Exclusion
# Dismiss the #1 recommended product
dismiss_target = results[0]["id"]
print(f"[ACTION] User dismisses product ID: {dismiss_target} ('{results[0]['productDisplayName']}')")

save_feedback(profile_id_a, dismiss_target, "dismiss", db_path=str(DB_PATH))
fb_after = get_feedback(profile_id_a, db_path=str(DB_PATH))
print(f"[SUCCESS] Profile A Dismissals list: {fb_after['dismissals']}")

# Re-run recommendation with the exact same query and constraints
rec_after_dismiss = recommend_personalized(
    profile_id=profile_id_a,
    query=query,
    constraints=constraints,
    top_k=5,
    model=embedding_model,
    catalog=catalog,
    embeddings=embeddings,
    catalog_ids=catalog_ids,
    db_path=str(DB_PATH),
    images_dir=str(IMAGES_DIR)
)

after_ids = [r["id"] for r in rec_after_dismiss["results"]]
assert dismiss_target not in after_ids, f"Dismissed product {dismiss_target} was not excluded!"

print(f"\n[VERIFIED] Dismissed product {dismiss_target} was strictly excluded from new recommendations:")
for r in rec_after_dismiss["results"]:
    print(f"  [{r['rank']}] Score: {r['score']:.4f} | ID: {r['id']} | {r['productDisplayName']} ({r['baseColour']})")


In [ ]:
# 6.3 Profile Switching & Profile Isolation Verification
# Switch active user context to Profile B
print(f"Switching active user context to Profile B ({profile_id_b})...")

# Profile B requests running shoes with active constraints
b_query = "breathable running sports shoes"
b_constraints = {
    "masterCategory": ["Footwear"],
    "usage": ["Sports"]
}

rec_b = recommend_personalized(
    profile_id=profile_id_b,
    query=b_query,
    constraints=b_constraints,
    top_k=5,
    model=embedding_model,
    catalog=catalog,
    embeddings=embeddings,
    catalog_ids=catalog_ids,
    db_path=str(DB_PATH),
    images_dir=str(IMAGES_DIR)
)

print(f"Profile B Results (Status: {rec_b['status']}):")
for r in rec_b["results"]:
    print(f"  [{r['rank']}] Score: {r['score']:.4f} | ID: {r['id']} | {r['productDisplayName']} ({r['masterCategory']} - {r['articleType']})")

# Verify isolation:
# 1. 100% of Profile B items are Footwear
# 2. None of Profile A's shirt likes or dismissals influenced Profile B
assert all(r["masterCategory"] == "Footwear" for r in rec_b["results"]), "Profile B constraints violated!"
print("\n[SUCCESS] Profile isolation verified: Profile B received sports footwear recommendations completely decoupled from Profile A.")


## 7. Explanations

In this section, we implement **Phase 4: Evidence-Based LLM Explanations & Factual Attribute Grounding** following `End_to_End PipeLine.pdf` (Step 8, Page 5):

1. **Strict Metadata Scope:** Sends ONLY returned product IDs (up to 5) and their recorded catalog attributes to Qwen. Never exposes the entire catalog to the model.
2. **Evidence-Based Factual Grounding:** Explanations must strictly state verified recorded metadata (Article Type, Color, Usage, Category, Season). Attribute claims are verified against the catalog row.
3. **Rejection of Unsupported Claims:** Explanations containing fabricated claims regarding fabric materials (e.g. 'cotton', 'silk'), fit ('perfect fit', 'will fit you perfectly'), quality ('premium', 'luxury'), or price/stock ('cheap', 'in stock') are strictly rejected.
4. **Deterministic Template Fallback:** If an LLM explanation fails or is rejected, a grounded template explanation is assigned. Recommendation results and ranking orders are 100% preserved.


In [ ]:
# 7.1 Evidence-Based Explanations & Grounding Validation
from engines.explanation_engine import (
    explain_items,
    validate_explanation,
    generate_template_explanation
)

# Use the top-5 recommendations generated in Section 6
rec_items = results[:5]
print(f"Generating evidence-based explanations for {len(rec_items)} recommended products...\n")

explanations_map = explain_items(
    selected_products=rec_items,
    active_constraints=constraints,
    profile_preferences=profile_a['preferences'],
    model=llm_model,
    tokenizer=llm_tokenizer
)

for rank, r in enumerate(rec_items, start=1):
    pid = r['id']
    expl = explanations_map.get(pid, 'No explanation.')
    print(f"[{rank}] ID: {pid:<6} | {r['productDisplayName']}")
    print(f"     Type: {r['articleType']} | Color: {r['baseColour']} | Usage: {r['usage']}")
    print(f"     Explanation: \"{expl}\"\n")


In [ ]:
# 7.2 Demonstration of Unsupported Claim Detection & Rejection
# Simulate an unsupported LLM hallucination
sample_hallucination = "This premium cotton shirt will fit you perfectly at the lowest price in stock."
is_valid, rejection_reason = validate_explanation(sample_hallucination, rec_items[0], constraints)

print(f"Testing Unsupported Claim Rejection:")
print(f"  Input Text:       \"{sample_hallucination}\"")
print(f"  Is Valid Grounded: {is_valid}")
print(f"  Rejection Reason:  {rejection_reason}")

# Fallback to deterministic grounded template
fallback_expl = generate_template_explanation(rec_items[0], constraints, profile_a['preferences'])
print(f"\nDeterministic Grounded Fallback:")
print(f"  \"{fallback_expl}\"")


## 8. Interactive Interface

> [!IMPORTANT]
> **Implementation Specification (Step 9 of `End_to_End_PipeLine.pdf`):**
> - Built entirely using Colab and Jupyter-supported `ipywidgets` (no external React, Streamlit, or web servers).
> - **Profile controls:** choose or create an anonymous profile, edit saved preferences, and clear active profile.
> - **Request controls:** text query input, category, color, usage dropdown filters, Recommend button, and Clear request button.
> - **Results display:** Top 5 recommendations with product images, metadata attributes, evidence-based explanations, and idempotent Like / Dismiss feedback controls.
> - **State handling:** working / searching indicator (disabling repeated submissions), zero-match handling with actionable relaxation suggestions, and unsupported attribute warning.


In [ ]:
# 8.1 Initialize and Display Interactive Fashion Recommender Prototype
from interactive_ui import FashionRecommenderApp

print("[INFO] Launching interactive fashion recommendation interface...")
app = FashionRecommenderApp()
app.render()


## 9. Formal Evaluation & 4-System Comparative Study

> [!IMPORTANT]
> **Evaluation Specification (Step 10 of `End_to_End_PipeLine.pdf`):**
> Compares retrieval under identical conditions across four distinct system variants:
> - **System A (Keyword baseline):** Token matching over product text after gold constraints.
> - **System B (Semantic retrieval):** `all-MiniLM-L6-v2` cosine similarity after gold constraints.
> - **System C (Personalized retrieval):** Variant B plus saved profile preferences and prior likes ($0.8 \times \text{query} + 0.2 \times \text{profile}$).
> - **System D (Complete Pipeline):** LLM-extracted constraints from natural language followed by Variant C and grounded explanations.
>
> Evaluated on 40 test queries across 4 groups:
> 1. Direct requests (10 queries)
> 2. Exclusions & multi-turn corrections (10 queries)
> 3. Paraphrased requests (10 queries)
> 4. Edge cases & unsupported attributes (10 queries)


In [ ]:
# 9.1 Execute Benchmark & Display 4-System Comparative Evaluation Table
from evaluation.run_evaluation import run_full_evaluation, format_evaluation_summary_table
import json
import pandas as pd
from IPython.display import display, Markdown

# Run or load benchmark evaluation metrics
metrics = run_full_evaluation()

print("\n" + "=" * 80)
print("STEP 10 RETRIEVAL EVALUATION RESULTS")
print("=" * 80)
display(Markdown(format_evaluation_summary_table(metrics)))

print("\nExtraction & Explanation Metrics:")
print(f"  • Extraction Exact Match Accuracy: {metrics['extraction_metrics']['exact_match_accuracy_pct']}%")
print(f"  • Valid JSON Output Rate: {metrics['extraction_metrics']['valid_json_rate_pct']}%")
print(f"  • Unsupported Field Handling Accuracy: {metrics['extraction_metrics']['unsupported_field_accuracy_pct']}%")
print(f"  • Explanation Grounding Compliance: {metrics['explanation_quality']['grounding_compliance_pct']}%")
print(f"  • Constraint Violation Rate (Target: 0.0%): {metrics['constraint_satisfaction']['violation_rates_pct']['System D (Full Pipeline)']}%")


## 10. Persistence, Runtime Recovery & Project Packaging

> [!IMPORTANT]
> **Implementation Specification (Steps 9 & 11 of `End_to_End_PipeLine.pdf`):**
> - **Persistence:** Uses SQLite Online Backup API (`conn.backup`) to create atomic snapshot `profiles_snapshot.sqlite`.
> - **Google Drive Recovery:** Workflow for saving and restoring profile databases and catalog artifacts across Colab runtime restarts.
> - **Integrity Verification:** Validates SHA-256 hashes against `catalog_manifest.json` and `embedding_manifest.json`.
> - **Project Export:** Generates standalone zip archive for external replication.


In [ ]:
# 10.1 SQLite Snapshot Backup & Integrity Verification
from backup_manager import create_database_backup, restore_database_backup, get_persistence_status

print("[INFO] Creating SQLite atomic database snapshot...")
backup_ok, backup_msg = create_database_backup()
print(f"Backup Result: {'[SUCCESS]' if backup_ok else '[FAILED]'} {backup_msg}")

# Verify persistence status
status = get_persistence_status()
print("\nPersistence Diagnostic Status:")
for k, v in status.items():
    print(f"  • {k}: exists={v['exists']}, valid={v['valid']}, size={v['size_bytes']} bytes")


In [ ]:
# 10.2 Artifact Checksum Verification & Export Archive Generation
import os
import json
import zipfile
from pathlib import Path
from prepare_catalog import compute_sha256
from embedding_engine import resolve_path

print("[INFO] Verifying artifact checksums against manifests...")
with open(resolve_path("fashion_project/artifacts/catalog_manifest.json"), "r") as f:
    cat_manifest = json.load(f)

for fname, meta in cat_manifest.get("files", {}).items():
    fp = resolve_path(f"fashion_project/artifacts/{fname}")
    if fp.exists():
        curr_hash = compute_sha256(fp)
        match = curr_hash == meta["sha256"]
        print(f"  [{'PASS' if match else 'FAIL'}] {fname}: hash match={match}")

# Package sanitized project export archive (excluding heavy raw data, pycache, and sqlite databases)
try:
    from scripts.export_package import package_project
    export_zip_path = package_project()
except ImportError:
    import sys
    sys.path.insert(0, str(resolve_path("fashion_project")))
    from scripts.export_package import package_project
    export_zip_path = package_project()

print(f"[SUCCESS] Export package ready at: {export_zip_path}")
